# Task 2 Predictions Demo

This notebook fits one recommender model on training ratings, generates top-10 movie recommendations for each user in `data/raw/ratings_test.csv`, and writes the completed submission CSV.

In [1]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from src.models.cold_start import BayesianColdStartRanker
from src.models.inference_router import RecommenderInferenceRouter
from src.models.item_knn_model import ItemKNNModel
from src.models.lightfm_model import LightFMHybridModel
from src.models.svd_model import SVDModel

/home/samuel/Documents/data-mining-recommender-systems/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Resolve repository root from notebook location.
project_root_path = Path.cwd()
if not (project_root_path / "src").exists():
    project_root_path = project_root_path.parent

# Use cleaned training/features by default.
train_ratings_path = project_root_path / "data" / "processed" / "ratings_train_cleaned.csv"
if not train_ratings_path.exists():
    train_ratings_path = project_root_path / "data" / "raw" / "ratings_train.csv"

movies_features_path = project_root_path / "data" / "processed" / "movies_cleaned.csv"
if not movies_features_path.exists():
    movies_features_path = project_root_path / "data" / "raw" / "movies.csv"

ratings_test_template_path = project_root_path / "data" / "raw" / "ratings_test.csv"
output_predictions_path = project_root_path / "data" / "processed" / "ratings_test_filled.csv"

train_ratings_dataframe = pd.read_csv(train_ratings_path)
movies_dataframe = pd.read_csv(movies_features_path)
ratings_test_dataframe = pd.read_csv(ratings_test_template_path)

print(f"Train rows: {len(train_ratings_dataframe)}")
print(f"Movies rows: {len(movies_dataframe)}")
print(f"Test users: {len(ratings_test_dataframe)}")

Train rows: 97801
Movies rows: 9742
Test users: 100


In [3]:
# Select the model for Task 2 export.
selected_model_name = "lightfm"  # Options: itemknn, svd, lightfm
top_n_recommendations = 10
minimum_personalization_interactions = 2

if selected_model_name == "itemknn":
    recommender_model = ItemKNNModel()
elif selected_model_name == "lightfm":
    recommender_model = LightFMHybridModel()
else:
    recommender_model = SVDModel()

# Fit the selected model.
if selected_model_name == "lightfm":
    recommender_model.fit(
        ratings_dataframe=train_ratings_dataframe,
        movies_dataframe=movies_dataframe,
    )
else:
    recommender_model.fit(ratings_dataframe=train_ratings_dataframe)

# Fit fallback ranker and route for cold-start users.
cold_start_ranker = BayesianColdStartRanker()
cold_start_ranker.fit(
    ratings_dataframe=train_ratings_dataframe,
    movies_dataframe=movies_dataframe,
)

inference_router = RecommenderInferenceRouter(
    trained_model=recommender_model,
    cold_start_ranker=cold_start_ranker,
    ratings_dataframe=train_ratings_dataframe,
    minimum_personalization_interactions=minimum_personalization_interactions,
)

print(f"Fitted model: {selected_model_name}")

Fitted model: lightfm


In [4]:
# Build top-10 recommendations for each user in the test template.
prediction_rows = []

for user_identifier in tqdm(ratings_test_dataframe["userId"].astype(int).tolist(), desc="Task2 users"):
    recommendation_result = inference_router.recommend_for_user(
        user_identifier=user_identifier,
        number_of_recommendations=top_n_recommendations,
    )

    recommended_movie_identifiers = [movie_id for movie_id, _ in recommendation_result.recommendations]

    # Guarantee exactly top_n_recommendations output values per user.
    if len(recommended_movie_identifiers) < top_n_recommendations:
        recommended_movie_identifiers = recommended_movie_identifiers + [None] * (
            top_n_recommendations - len(recommended_movie_identifiers)
        )

    prediction_row = {"userId": user_identifier}
    for recommendation_index in range(top_n_recommendations):
        column_name = f"recommendation{recommendation_index + 1}"
        prediction_row[column_name] = recommended_movie_identifiers[recommendation_index]
    prediction_rows.append(prediction_row)

predictions_dataframe = pd.DataFrame(prediction_rows)
predictions_dataframe.head()

,userId,recommendation1,recommendation2,recommendation3,recommendation4,recommendation5,recommendation6,recommendation7,recommendation8,recommendation9,recommendation10
0,3,161918,1200,60471,27793,26849,36509,610,3930,187595,8830
1,7,87520,87232,88140,71999,101076,77561,52722,79132,122886,103042
2,11,8481,3633,8118,2404,3452,1249,3197,459,87232,88140
3,25,87520,122886,103042,110102,111364,101864,102445,102880,103228,106002
4,30,87520,85261,110102,111364,101864,102445,102880,103228,106002,106487


In [5]:
# Save completed Task 2 CSV file.
output_predictions_path.parent.mkdir(parents=True, exist_ok=True)
predictions_dataframe.to_csv(output_predictions_path, index=False)

print(f"Saved predictions to: {output_predictions_path}")
print(f"Output rows: {len(predictions_dataframe)}")
print("Output columns:", list(predictions_dataframe.columns))

Saved predictions to: /home/samuel/Documents/data-mining-recommender-systems/data/processed/ratings_test_filled.csv
Output rows: 100
Output columns: ['userId', 'recommendation1', 'recommendation2', 'recommendation3', 'recommendation4', 'recommendation5', 'recommendation6', 'recommendation7', 'recommendation8', 'recommendation9', 'recommendation10']


In [6]:
# Quick consistency checks for submission shape.
assert len(predictions_dataframe) == len(ratings_test_dataframe), "Row count mismatch with ratings_test.csv"
expected_columns = ["userId"] + [f"recommendation{i}" for i in range(1, 11)]
assert list(predictions_dataframe.columns) == expected_columns, "Column mismatch for submission file"

predictions_dataframe.isna().sum()

userId              0
recommendation1     0
recommendation2     0
recommendation3     0
recommendation4     0
recommendation5     0
recommendation6     0
recommendation7     0
recommendation8     0
recommendation9     0
recommendation10    0
dtype: int64